# 03 — Fine-tuning da StarNet-S4 (baseline)

Treina a StarNet-S4 (MA et al., 2024) a partir dos pesos ImageNet-1K oficiais no dataset
`cerrado_df_v1`, com `configs/experiments/starnet_s4_ft.yaml`, **idêntica** à do GmNet-S3
exceto pelo modelo (regra 7; conferido por `tests/test_configs.py`).

Desenho (b): GmNet-S3 completo vs. StarNet-S4. As arquiteturas diferem em mais do que o
gating (larguras/profundidades, BN e *layer scale*, receita de treino no ImageNet), então a
comparação **não isola** o efeito do gating.

**Antes**: rode o `00_setup_colab.ipynb` no mesmo ambiente. Use um ambiente com **GPU**.

- Pesos: baixados do release oficial para `<drive_root>/pretrained/` se não existirem, com
  SHA-256 conferido, seguidos de um teste de sanidade em 200 imagens do ImageNet.
- Checkpoints e resultados: como no notebook 02, em `<drive_root>/checkpoints|results/starnet_s4_ft/seed_<n>/`.
- Resultados para o artigo: só no modo `git`, sem alterações não commitadas.

In [ ]:
CONFIG = "configs/experiments/starnet_s4_ft.yaml"
RUN_TESTS = True  # roda a suíte de testes (incluindo os de PyTorch) antes de treinar

## 1. Ambiente

In [ ]:
import importlib
import sys
from pathlib import Path

import torch
import yaml

REPO_DIR = Path("/content/ProjetoFinalRNAP_UnB2026")
assert REPO_DIR.exists(), "Rode antes o 00_setup_colab.ipynb neste ambiente."

# O kernel em execução não lê o .pth da instalação editável (ver notebook 00).
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

from cerrado_flora.utils import run_metadata

paths = yaml.safe_load((REPO_DIR / "configs/paths.yaml").read_text())
cfg = yaml.safe_load((REPO_DIR / CONFIG).read_text())
pretrained_cfg = yaml.safe_load((REPO_DIR / "configs/pretrained.yaml").read_text())

drive_root = Path(paths["drive_root"])
assert drive_root.exists(), "Google Drive não montado (seção 1 do notebook 00)."
images_root = Path(paths["local_cache"]) / cfg["dataset"]["name"]
assert (images_root / "images").exists(), "Dataset ausente: rode a seção 5 do notebook 00."
manifest = REPO_DIR / cfg["dataset"]["manifest"]

meta = run_metadata.collect(REPO_DIR)
print("commit:", meta["git_commit"], "| alterações não commitadas:", meta["git_dirty"])
if meta["git_commit"] is None or meta["git_dirty"]:
    print("AVISO: sem commit limpo; estes resultados não devem entrar no artigo (modo git).")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
assert torch.cuda.is_available(), "Selecione um ambiente com GPU."

## 2. Testes

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rs", "tests"], cwd=REPO_DIR,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0, "testes falharam"

## 3. Pesos pré-treinados

In [ ]:
from cerrado_flora.data.download import download
from cerrado_flora.data.inaturalist import make_session
from cerrado_flora.models.factory import verify_checkpoint

wcfg = pretrained_cfg[cfg["model"]["pretrained"]]
weights = drive_root / paths["drive_subdirs"]["pretrained"] / wcfg["file"]
if not weights.exists():
    err = download(make_session("cerrado-flora/0.1 (academic research)"), wcfg["url"], weights)
    assert err is None, err
verify_checkpoint(weights, wcfg["sha256"])
print("pesos verificados:", weights)

Teste de sanidade: os pesos com a cabeça original de 1000 classes em 200 imagens do ImageNet
(mesmo protocolo usado para o GmNet-S3, que deu 86,5% de top-1). Pesos carregados na
arquitetura errada dão ~0%.

In [ ]:
from cerrado_flora.evaluation.imagenet_sanity import download_sample, imagenet_topk
from cerrado_flora.models.factory import build_model

samples = download_sample("/content/cache/imagenet_sample")
sanity = imagenet_topk(build_model(cfg["model"], 1000, weights), samples, torch.device("cuda"))
print("ImageNet (amostra):", sanity)
assert sanity["top1"] > 0.5, "pesos possivelmente carregados de forma incorreta"

## 4. Treino e avaliação

In [ ]:
from cerrado_flora.training.train import run_experiment

exp = cfg["name"]
results = {}
for seed in cfg["seeds"]:
    results[seed] = run_experiment(
        cfg, seed, manifest, images_root, weights,
        ckpt_dir=drive_root / paths["drive_subdirs"]["checkpoints"] / exp / f"seed_{seed}",
        results_dir=drive_root / paths["drive_subdirs"]["results"] / exp / f"seed_{seed}",
    )

## 5. Resultados

In [ ]:
import pandas as pd

rows = []
for seed, r in results.items():
    for split in ("val", "test"):
        for level in ("photo", "observation"):
            rows.append({"seed": seed, "divisao": split, "nivel": level,
                         **{k: r[split][level][k] for k in ("top1", "macro_f1", "balanced_accuracy", "n")},
                         "melhor_epoca": r["best_epoch"]})
pd.DataFrame(rows).round(4)

In [ ]:
import matplotlib.pyplot as plt

seed = cfg["seeds"][0]
res_dir = drive_root / paths["drive_subdirs"]["results"] / exp / f"seed_{seed}"
h = pd.read_csv(res_dir / "history.csv")
blue, orange = "#2a78d6", "#eb6834"  # paleta categórica validada

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(h["epoch"], h["train_loss"], color=blue, lw=2, label="treino (com label smoothing)")
a1.plot(h["epoch"], h["val_loss"], color=orange, lw=2, label="validação")
a1.set_title("Perda", loc="left")
a2.plot(h["epoch"], h["val_macro_f1"], color=blue, lw=2, label="macro-F1 (foto)")
a2.plot(h["epoch"], h["val_top1"], color=orange, lw=2, label="top-1 (foto)")
a2.axvline(results[seed]["best_epoch"], color="#a3a29d", lw=1, ls="--")
a2.set_title("Validação", loc="left")
for a in (a1, a2):
    a.set_xlabel("época")
    a.grid(color="#e5e4e0", lw=0.6)
    a.set_axisbelow(True)
    for side in ("top", "right"):
        a.spines[side].set_visible(False)
    a.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"tempo médio por época: {h['epoch_time_s'].mean():.0f}s")

F1 por classe no teste (por foto), da pior para a melhor:

In [ ]:
species = pd.read_csv(manifest.parent / "species.csv").dropna(subset=["class_index"])
names = species.set_index(species["class_index"].astype(int))["scientific_name"]
pc = pd.read_csv(res_dir / "per_class_test.csv", index_col="class_index")
pc.insert(0, "especie", names)
pc.sort_values("f1").round(3)